# Extração de embedding intermediário

Notebook pedagógico que percorre **etapa a etapa** o pipeline de [`embeddings.py`](../src/embeddings.py):

**texto → tokenização → MiniLM (`last_hidden_state`) → filtro de especiais/padding → matriz $F \in \mathbb{R}^{T \times D}$**

Cada **linha** de $F$ é o embedding contextualizado de um subtoken; $D=384$ no `all-MiniLM-L6-v2`. Usamos token-level (não pooling de sentença) porque o trabalho **poda linhas** de $F$ antes do classificador downstream.

Próximo passo: [`compressao_tokens.ipynb`](compressao_tokens.ipynb) (poda) → [`avaliacao_metricas.ipynb`](avaliacao_metricas.ipynb) (métricas) → [`experimento.ipynb`](experimento.ipynb) (grade) → [`trabalho_final.ipynb`](trabalho_final.ipynb) (resultados e validação).

In [ ]:
import os

import numpy as np

import dados
import embeddings
from constantes import MAX_AMOSTRAS, MAX_TOKENS, N_POR_CLASSE, SEED

print("Diretório de trabalho:", os.getcwd())

## 1. Corpus — uma review de exemplo

Carregamos o subconjunto IMDb balanceado (mesmos defaults de [`constantes.py`](constantes.py)) e usamos a primeira review como exemplo.

In [ ]:
textos, rotulos = dados.carregar_imdb(
    n_por_classe=N_POR_CLASSE, max_amostras=MAX_AMOSTRAS, semente_split=SEED,
)
texto_exemplo = textos[0]
print(f"Corpus: {len(textos)} reviews | exemplo: rótulo={rotulos[0]}")
print(f"Trecho: {texto_exemplo[:200]}...")

## 2. Modelo — carregar tokenizer e encoder

`carregar_modelo` usa `AutoModel` (não `SentenceTransformer`) porque precisamos de **`last_hidden_state`**: um vetor **por token**, não um único vetor por sentença.

In [ ]:
tokenizer, modelo = embeddings.carregar_modelo()
print("Modelo:", embeddings.MODELO_PADRAO)
print("Modo eval:", not modelo.training)

## 3. Tokenização

`max_tokens` limita posições do modelo (subtokens), não palavras. `truncation=True` corta reviews longas.

In [ ]:
entradas = embeddings.tokenizar_texto(texto_exemplo, tokenizer, max_tokens=MAX_TOKENS)
tokens = embeddings.obter_tokens(tokenizer, entradas)

print(f"Número de posições L (com [CLS]/[SEP]): {len(tokens)}")
print("Primeiros 15 tokens:", tokens[:15])
print("Últimos 5 tokens:", tokens[-5:])

## 4. Encoder — hidden states

`extrair_hidden_states` executa o forward pass com `torch.no_grad()` (inferência pura, sem grafo de gradientes) e retorna `last_hidden_state[0]` com shape $(L, D)$.

$L$ ainda inclui `[CLS]`, `[SEP]` e possível padding.

In [ ]:
hidden = embeddings.extrair_hidden_states(modelo, entradas)
print("hidden shape (L x D):", hidden.shape)
print("D (dimensão embedding):", hidden.shape[1], "— esperado 384")

## 5. Filtro — tokens de conteúdo

Removemos `[CLS]`, `[SEP]` e padding. O resultado é $F \in \mathbb{R}^{T' \times D}$, onde $T' \leq L$.

In [ ]:
indices = embeddings.indices_tokens_conteudo(tokenizer, entradas)
F = embeddings.selecionar_tokens_conteudo(hidden, indices)
tokens_conteudo = [tokens[i] for i in indices]

print(f"Posições de conteúdo: {len(indices)} (L={len(tokens)} → T'={F.shape[0]})")
print("Primeiros tokens de conteúdo:", tokens_conteudo[:10])

## 6. Atalho — `texto_para_embedding` e debug

A função usada em [`experimento.py`](../src/experimento.py) encapsula os passos acima. `texto_para_embedding_debug` expõe os intermediários para inspeção.

In [ ]:
F_atalho = embeddings.texto_para_embedding(texto_exemplo, max_tokens=MAX_TOKENS)
dbg = embeddings.texto_para_embedding_debug(texto_exemplo, max_tokens=MAX_TOKENS)

print("F manual vs atalho:", np.allclose(F, F_atalho))
print("F_atalho shape:", F_atalho.shape)
print("Chaves debug:", list(dbg.keys()))

## 7. Estatística — $T$ médio no corpus

Calculamos $\bar{T}$ sobre uma amostra do corpus (10 reviews para não demorar; a grade usa todas).

In [ ]:
n_amostra = min(10, len(textos))
ts = [
    embeddings.texto_para_embedding(t, max_tokens=MAX_TOKENS).shape[0]
    for t in textos[:n_amostra]
]
print(f"T por review (primeiras {n_amostra}): min={min(ts)}, max={max(ts)}, média={np.mean(ts):.1f}")